# 使用腾讯混元API获取最新技术资讯

本笔记本演示如何使用腾讯混元(Hunyuan)大模型的联网搜索功能，获取感兴趣领域的最新资讯。

**参考文档**: [腾讯混元OpenAI兼容接口文档](https://cloud.tencent.com/document/product/1729/111007)

## 步骤1: 导入必要的库

In [18]:
import json
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import re

print("✓ 库导入成功")

✓ 库导入成功


## 步骤2: 加载环境变量和配置

从 `.env` 文件中加载API密钥

In [19]:
# 加载环境变量
load_dotenv()

# 获取API凭证
HUNYUAN_API_KEY = os.getenv('HUNYUAN_API_KEY')

# API配置
API_URL = "https://api.hunyuan.cloud.tencent.com/v1/chat/completions"

# 验证API密钥是否加载成功
if HUNYUAN_API_KEY:
    print(f"✓ API密钥已加载 (长度: {len(HUNYUAN_API_KEY)} 字符)")
else:
    print("✗ 警告: API密钥未找到，请检查.env文件")

✓ API密钥已加载 (长度: 51 字符)


## 步骤3: 定义感兴趣的技术领域

自定义你想要获取资讯的技术领域

In [20]:
# 定义感兴趣的技术领域
TECH_FIELDS = [
    "自动驾驶",
    "具身智能",
    "大模型",
    "人工智能"
]

# 定义重点关注的主题
FOCUS_TOPICS = [
    "特斯拉FSD",
    "人形机器人",
    "AI大模型突破",
    "行业重要动态"
]

print(f"关注领域: {', '.join(TECH_FIELDS)}")
print(f"重点主题: {', '.join(FOCUS_TOPICS)}")

关注领域: 自动驾驶, 具身智能, 大模型, 人工智能
重点主题: 特斯拉FSD, 人形机器人, AI大模型突破, 行业重要动态


## 步骤4: 创建智能搜索Prompt

构建一个专业的prompt，指导AI进行联网搜索并返回结构化数据

In [21]:
def create_tech_news_prompt(fields, focus_topics, num_items_per_field=2):
    """
    创建专业的技术资讯搜索prompt
    
    参数:
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        num_items_per_field: 每个领域返回的新闻数量
    """
    current_date = datetime.now().strftime("%Y-%m-%d")
    fields_str = "、".join(fields)
    topics_str = "、".join(focus_topics)
    
    prompt = f"""你是一个专业的技术资讯分析师。今天是{current_date}，请使用联网搜索功能获取以下技术领域的最新动态：

搜索领域：{fields_str}
重点关注：{topics_str}

请严格按照以下JSON格式返回数据，不要有任何额外的解释或文本：

{{
  "search_date": "{current_date}",
  "summary": "今日技术领域总体趋势的简要描述",
  "news_items": [
    {{
      "id": 1,
      "category": "技术领域分类",
      "title": "新闻标题",
      "content": "详细内容摘要(150-200字)",
      "key_entities": ["相关公司", "相关人物", "技术名称"],
      "source_type": "信息来源平台",
      "source_link": "信息来源链接",
      "significance": "高/中/低",
      "timestamp": "信息发布时间或发现时间[YYYY-mm-dd HH:MM]"
    }}
  ]
}}

具体要求：
1. 每个技术领域提供{num_items_per_field}-{num_items_per_field+1}条最重要的最新信息
2. 按重要性排序，最重要的信息在前
3. 确保信息准确且是最新的（24小时内）
4. 关键实体要准确识别
5. significance评估基于技术突破性、行业影响力
6. source_type注明如：微博、公众号、科技媒体、学术平台等

现在开始搜索并返回JSON数据："""
    
    return prompt

# 创建prompt
prompt = create_tech_news_prompt(TECH_FIELDS, FOCUS_TOPICS)
print("✓ Prompt已创建")
print(f"\nPrompt预览 (前300字符):\n{prompt[:300]}...")

✓ Prompt已创建

Prompt预览 (前300字符):
你是一个专业的技术资讯分析师。今天是2025-11-10，请使用联网搜索功能获取以下技术领域的最新动态：

搜索领域：自动驾驶、具身智能、大模型、人工智能
重点关注：特斯拉FSD、人形机器人、AI大模型突破、行业重要动态

请严格按照以下JSON格式返回数据，不要有任何额外的解释或文本：

{
  "search_date": "2025-11-10",
  "summary": "今日技术领域总体趋势的简要描述",
  "news_items": [
    {
      "id": 1,
      "category": "技术领域分类",
      "title": "新闻标题",...


## 步骤5: 调用API获取最新资讯

使用腾讯混元API的联网搜索功能获取实时信息

In [22]:
def get_latest_news(prompt, api_key, model="hunyuan-turbo", temperature=0.3):
    """
    调用腾讯混元API获取最新资讯
    
    参数:
        prompt: 搜索提示词
        api_key: API密钥
        model: 模型版本
        temperature: 温度参数 (0-2)，越低输出越稳定
    
    返回:
        API响应的原始文本
    """
    # 构建请求头
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }
    
    # 构建请求体
    body = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,  # 非流式响应，获取完整JSON
        "temperature": temperature,  # 较低温度保证输出稳定性
        "enable_enhancement": True,  # 启用联网搜索增强
        "search_info": True  # 返回搜索信息
    }
    
    try:
        print("正在调用API...")
        response = requests.post(API_URL, json=body, headers=headers, timeout=60)
        
        # 检查HTTP状态码
        if response.status_code != 200:
            print(f"✗ API返回错误状态码: {response.status_code}")
            print(f"响应内容: {response.text}")
            return None
        
        print("✓ API调用成功")
        
        # 解析响应
        data = response.json()
        
        # 提取内容
        if 'choices' in data and len(data['choices']) > 0:
            content = data['choices'][0]['message']['content']
            print(f"✓ 获取到响应内容 (长度: {len(content)} 字符)")
            return content
        else:
            print("✗ 响应格式异常")
            return None
            
    except requests.exceptions.Timeout:
        print("✗ API调用超时")
        return None
    except Exception as e:
        print(f"✗ API调用错误: {e}")
        return None

# 调用API
raw_response = get_latest_news(prompt, HUNYUAN_API_KEY)

if raw_response:
    print(f"\n响应预览 (前500字符):\n{raw_response[:500]}...")

正在调用API...
✓ API调用成功
✓ 获取到响应内容 (长度: 3911 字符)

响应预览 (前500字符):
{
  "search_date": "2025-11-10",
  "summary": "今日技术领域聚焦自动驾驶、具身智能、大模型及人工智能最新进展。自动驾驶方面，特斯拉Cybercab在进博会首秀，无方向盘设计引爆关注；华为发布L3/L4商用路线图，2026年高速L3、2027年城区L4规模商用目标引发行业热议。具身智能领域，北京人形机器人与拜耳医药合作探索制药场景应用，中关村具身智能大赛30强诞生，推动技术向实用化落地。大模型方面，科大讯飞星火X1.5、文心ERNIE-5.0-Preview等国产模型升级，性能对标国际主流；阶跃星辰开源音频编辑大模型，实现零样本语音克隆。人工智能行业，Kosmos AI系统完成科研循环，Meta回购AI蛋白质团队，谷歌Nano Banana 2推演微积分，技术突破与应用拓展同步推进。",
  "news_items": [
    {
      "id": 1,
      "category": "自动驾驶",
      "title": "特斯拉Cybercab在进博会首秀，无方向盘设计成焦点",
      "content": "...


## 步骤6: 解析JSON响应

从AI返回的文本中提取结构化的JSON数据

In [23]:
def extract_json_from_response(response_text):
    """
    从响应文本中提取JSON数据
    
    尝试多种方法提取和解析JSON:
    1. 查找JSON代码块 (```json ... ```)
    2. 直接查找JSON结构 ({ ... })
    3. 清理文本后解析
    """
    if not response_text:
        return {"error": "响应为空"}
    
    # 方法1: 查找Markdown格式的JSON代码块
    json_block_match = re.search(r'```json\s*(.+?)\s*```', response_text, re.DOTALL)
    if json_block_match:
        try:
            json_str = json_block_match.group(1)
            print("✓ 从代码块中提取JSON")
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"⚠ 代码块JSON解析失败: {e}")
    
    # 方法2: 直接查找JSON结构
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if json_match:
        try:
            json_str = json_match.group()
            print("✓ 直接提取JSON结构")
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"⚠ 直接提取JSON解析失败: {e}")
    
    # 方法3: 清理并尝试解析整个文本
    try:
        clean_text = response_text.strip()
        # 移除可能的markdown标记
        if clean_text.startswith('```json'):
            clean_text = clean_text[7:]
        if clean_text.startswith('```'):
            clean_text = clean_text[3:]
        if clean_text.endswith('```'):
            clean_text = clean_text[:-3]
        
        print("✓ 尝试清理后解析")
        return json.loads(clean_text.strip())
    except json.JSONDecodeError as e:
        print(f"✗ JSON解析失败: {e}")
        return {
            "error": "无法解析JSON",
            "parse_error": str(e),
            "raw_response": response_text[:1000]  # 只保留前1000字符
        }

# 解析响应
if raw_response:
    news_data = extract_json_from_response(raw_response)
    
    if "error" in news_data:
        print(f"\n✗ 解析失败: {news_data.get('error')}")
    else:
        print(f"\n✓ JSON解析成功")
        print(f"  - 搜索日期: {news_data.get('search_date', 'N/A')}")
        print(f"  - 新闻条目数: {len(news_data.get('news_items', []))}")
else:
    print("无法解析: 未获取到有效响应")
    news_data = None

✓ 直接提取JSON结构

✓ JSON解析成功
  - 搜索日期: 2025-11-10
  - 新闻条目数: 8


## 步骤7: 可视化展示资讯摘要

以友好的格式展示获取到的技术资讯

In [24]:
def display_news_summary(news_data):
    """
    以友好格式显示新闻摘要
    """
    if not news_data or "error" in news_data:
        print("✗ 无法显示：数据无效或包含错误")
        if "error" in news_data:
            print(f"错误信息: {news_data['error']}")
        return
    
    # 显示标题
    print("\n" + "="*80)
    print(f"{'技术动态简报':^76}")
    print(f"{'日期: ' + news_data.get('search_date', 'N/A'):^76}")
    print("="*80)
    
    # 显示总体趋势
    summary = news_data.get('summary', 'N/A')
    print(f"\n📊 总体趋势:\n{summary}")
    print("\n" + "-"*80)
    
    # 显示新闻条目
    news_items = news_data.get('news_items', [])
    print(f"\n📰 共找到 {len(news_items)} 条技术资讯:\n")
    
    for i, item in enumerate(news_items, 1):
        # 重要性标记
        significance = item.get('significance', '中')
        sig_icon = {'高': '🔴', '中': '🟡', '低': '🟢'}.get(significance, '⚪')
        
        print(f"\n{i}. {sig_icon} [{item.get('category', 'N/A')}] {item.get('title', 'N/A')}")
        print(f"\n   💬 内容摘要:")
        print(f"   {item.get('content', 'N/A')}")
        
        # 关键实体
        entities = item.get('key_entities', [])
        if entities:
            print(f"\n   🏷️  关键实体: {', '.join(entities)}")
        
        # 元信息
        print(f"   📍 来源: {item.get('source_type', 'N/A')}")
        print(f"   ⏰ 时间: {item.get('timestamp', 'N/A')}")
        print(f"   ⭐ 重要性: {significance}")
        
        print("\n" + "-"*80)

# 显示新闻摘要
if news_data:
    display_news_summary(news_data)


                                   技术动态简报                                   
                               日期: 2025-11-10                               

📊 总体趋势:
今日技术领域聚焦自动驾驶、具身智能、大模型及人工智能最新进展。自动驾驶方面，特斯拉Cybercab在进博会首秀，无方向盘设计引爆关注；华为发布L3/L4商用路线图，2026年高速L3、2027年城区L4规模商用目标引发行业热议。具身智能领域，北京人形机器人与拜耳医药合作探索制药场景应用，中关村具身智能大赛30强诞生，推动技术向实用化落地。大模型方面，科大讯飞星火X1.5、文心ERNIE-5.0-Preview等国产模型升级，性能对标国际主流；阶跃星辰开源音频编辑大模型，实现零样本语音克隆。人工智能行业，Kosmos AI系统完成科研循环，Meta回购AI蛋白质团队，谷歌Nano Banana 2推演微积分，技术突破与应用拓展同步推进。

--------------------------------------------------------------------------------

📰 共找到 8 条技术资讯:


1. 🔴 [自动驾驶] 特斯拉Cybercab在进博会首秀，无方向盘设计成焦点

   💬 内容摘要:
   第八届中国国际进口博览会上，特斯拉带来Cybercab亚太地区首次亮相。该车采用无方向盘、无脚踏板设计，搭载Tesla Vision视觉处理系统+端到端神经网络，无需激光雷达即可实现无人驾驶。马斯克表示，Cybercab是特斯拉‘秘密宏图’第四篇章的重要载体，旨在推动可持续富足社会实现。

   🏷️  关键实体: 特斯拉, Cybercab, 进博会
   📍 来源: 新闻网站
   ⏰ 时间: 2025-11-05 15:30
   ⭐ 重要性: 高

--------------------------------------------------------------------------------

2. 🔴 [自动驾驶] 华为发布L3/L4商用路线图，2026年高速L3、2027年城区L4

## 步骤8: 保存数据到文件

将获取的资讯保存为JSON文件，方便后续分析和使用

In [25]:
def save_news_to_file(news_data, filename=None):
    """
    保存新闻数据到JSON文件
    
    参数:
        news_data: 新闻数据字典
        filename: 文件名（可选），默认使用时间戳命名
    
    返回:
        保存的文件路径
    """
    if not news_data or "error" in news_data:
        print("✗ 数据无效，无法保存")
        return None
    
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"tech_news_hunyuan_{timestamp}.json"
    
    try:
        # 保存JSON文件
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(news_data, f, ensure_ascii=False, indent=2)
        
        file_size = os.path.getsize(filename)
        print(f"✓ 数据已保存到: {filename}")
        print(f"  文件大小: {file_size} 字节")
        
        return filename
    except Exception as e:
        print(f"✗ 保存文件失败: {e}")
        return None

# 保存数据
if news_data:
    saved_file = save_news_to_file(news_data)
    if saved_file:
        print(f"\n可以使用以下代码重新加载数据:")
        print(f"with open('{saved_file}', 'r', encoding='utf-8') as f:")
        print(f"    news_data = json.load(f)")

✓ 数据已保存到: tech_news_hunyuan_20251110_092817.json
  文件大小: 6915 字节

可以使用以下代码重新加载数据:
with open('tech_news_hunyuan_20251110_092817.json', 'r', encoding='utf-8') as f:
    news_data = json.load(f)


## 步骤9: 数据分析和统计

对获取的资讯进行简单的统计分析

In [26]:
def analyze_news_data(news_data):
    """
    分析新闻数据，生成统计报告
    """
    if not news_data or "error" in news_data:
        print("无法分析：数据无效")
        return
    
    news_items = news_data.get('news_items', [])
    
    if not news_items:
        print("无新闻数据可供分析")
        return
    
    print("\n" + "="*80)
    print(f"{'数据统计分析':^76}")
    print("="*80)
    
    # 1. 按类别统计
    category_count = {}
    for item in news_items:
        category = item.get('category', '未分类')
        category_count[category] = category_count.get(category, 0) + 1
    
    print("\n📊 按类别统计:")
    for category, count in sorted(category_count.items(), key=lambda x: x[1], reverse=True):
        print(f"   {category}: {count} 条")
    
    # 2. 按重要性统计
    significance_count = {'高': 0, '中': 0, '低': 0}
    for item in news_items:
        sig = item.get('significance', '中')
        if sig in significance_count:
            significance_count[sig] += 1
    
    print("\n⭐ 按重要性统计:")
    for sig, count in significance_count.items():
        print(f"   {sig}: {count} 条")
    
    # 3. 按来源统计
    source_count = {}
    for item in news_items:
        source = item.get('source_type', '未知来源')
        source_count[source] = source_count.get(source, 0) + 1
    
    print("\n📍 按来源统计:")
    for source, count in sorted(source_count.items(), key=lambda x: x[1], reverse=True):
        print(f"   {source}: {count} 条")
    
    # 4. 关键实体统计
    all_entities = []
    for item in news_items:
        all_entities.extend(item.get('key_entities', []))
    
    entity_count = {}
    for entity in all_entities:
        entity_count[entity] = entity_count.get(entity, 0) + 1
    
    print("\n🏷️  热门关键实体 (Top 10):")
    top_entities = sorted(entity_count.items(), key=lambda x: x[1], reverse=True)[:10]
    for entity, count in top_entities:
        print(f"   {entity}: {count} 次")
    
    print("\n" + "="*80)

# 执行数据分析
if news_data:
    analyze_news_data(news_data)


                                   数据统计分析                                   

📊 按类别统计:
   大模型: 3 条
   自动驾驶: 2 条
   具身智能: 2 条
   人工智能: 1 条

⭐ 按重要性统计:
   高: 5 条
   中: 3 条
   低: 0 条

📍 按来源统计:
   新闻网站: 5 条
   公众号: 3 条

🏷️  热门关键实体 (Top 10):
   特斯拉: 1 次
   Cybercab: 1 次
   进博会: 1 次
   华为: 1 次
   乾崑智驾生态开放平台: 1 次
   自动驾驶出行生态论坛: 1 次
   北京人形机器人: 1 次
   拜耳医药: 1 次
   天工机器人: 1 次
   中关村具身智能大赛: 1 次



## 步骤10: 完整流程封装

将以上步骤封装成一个完整的函数，方便重复使用

In [27]:
def acquire_latest_news(fields, focus_topics, save_to_file=True, show_analysis=True):
    """
    完整的新闻获取流程
    
    参数:
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        save_to_file: 是否保存到文件
        show_analysis: 是否显示统计分析
    
    返回:
        新闻数据字典
    """
    print("\n" + "="*80)
    print(f"{'开始获取最新技术资讯 (腾讯混元)':^76}")
    print("="*80)
    
    # 1. 创建prompt
    print("\n[1/5] 创建搜索prompt...")
    prompt = create_tech_news_prompt(fields, focus_topics)
    
    # 2. 调用API
    print("\n[2/5] 调用API进行联网搜索...")
    raw_response = get_latest_news(prompt, HUNYUAN_API_KEY)
    
    if not raw_response:
        print("\n✗ 流程终止：API调用失败")
        return None
    
    # 3. 解析JSON
    print("\n[3/5] 解析响应数据...")
    news_data = extract_json_from_response(raw_response)
    
    if "error" in news_data:
        print("\n✗ 流程终止：数据解析失败")
        return None
    
    # 4. 显示结果
    print("\n[4/5] 显示新闻摘要...")
    display_news_summary(news_data)
    
    # 5. 保存文件
    if save_to_file:
        print("\n[5/5] 保存数据到文件...")
        save_news_to_file(news_data)
    
    # 6. 统计分析（可选）
    if show_analysis:
        analyze_news_data(news_data)
    
    print("\n" + "="*80)
    print(f"{'流程完成！':^76}")
    print("="*80)
    
    return news_data

# 示例：使用封装的函数
# news_result = acquire_latest_news(
#     fields=["人工智能", "机器学习", "深度学习"],
#     focus_topics=["GPT", "Transformer", "计算机视觉"],
#     save_to_file=True,
#     show_analysis=True
# )

## 总结

本笔记本演示了如何使用腾讯混元API的联网搜索功能获取最新技术资讯，包括：

1. ✅ 环境配置和API密钥加载
2. ✅ 自定义技术领域和关注主题
3. ✅ 构建智能搜索prompt
4. ✅ 调用API进行联网搜索
5. ✅ 解析和验证JSON响应
6. ✅ 可视化展示资讯
7. ✅ 保存数据到文件
8. ✅ 数据统计分析
9. ✅ 完整流程封装

### 使用建议

- 根据需要修改 `TECH_FIELDS` 和 `FOCUS_TOPICS` 来定制你的资讯关注点
- 调整 `temperature` 参数控制输出的随机性（0-2之间，越低越稳定）
- 可以使用 `acquire_latest_news()` 函数快速获取资讯
- 保存的JSON文件可以用于后续的数据分析和可视化

### 下一步

- 可以添加定时任务，定期获取资讯
- 可以集成更多数据分析和可视化功能
- 可以添加资讯推送通知功能
- 可以构建知识图谱，追踪技术发展趋势